# ☁️ Module 7.4: AWS Deployment

**Goal**: Deploy an MLFlow model to AWS using ECR and SageMaker/EC2.

**Prerequisites**:
- AWS account with free tier
- AWS CLI configured (`aws configure`)
- Docker Desktop running

---

## ⚠️ Cost Warning

> **SageMaker endpoints incur charges even when idle.** Always run the cleanup cell at the end of this notebook to delete endpoints. The free tier covers limited usage but be careful with long-running endpoints.

## Step 0: Verify AWS CLI

In [ ]:
import subprocess

try:
    result = subprocess.run(["aws", "--version"], capture_output=True, text=True)
    print(f"✅ {result.stdout.strip()}")
    
    # Check credentials
    result = subprocess.run(["aws", "sts", "get-caller-identity"], capture_output=True, text=True)
    if result.returncode == 0:
        import json
        identity = json.loads(result.stdout)
        print(f"✅ Authenticated as: {identity.get('Arn', 'unknown')}")
    else:
        print("❌ AWS credentials not configured. Run: aws configure")
except FileNotFoundError:
    print("❌ AWS CLI not found. Install it first:")
    print("   https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html")

## Step 1: Prepare the Model

We'll use a model from Module 7.3 or train a new one.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

mlflow.autolog(disable=True)
mlflow.set_experiment("07_aws_deployment")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="aws_deploy_model"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(model, "model", signature=signature, input_example=X_train[:2])
    run_id = mlflow.active_run().info.run_id

print(f"✅ Model ready. Run ID: {run_id}")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## Step 2: Build & Push Docker Image to AWS ECR

AWS Elastic Container Registry (ECR) stores Docker images. We need to:
1. Build the Docker image
2. Create an ECR repository
3. Push the image to ECR

In [ ]:
import boto3
import json

# Get AWS account info
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
region = boto3.session.Session().region_name or "us-east-1"

IMAGE_NAME = "wine-quality-model"
ECR_REPO = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{IMAGE_NAME}"

print(f"AWS Account: {account_id}")
print(f"Region: {region}")
print(f"ECR Repository: {ECR_REPO}")

### 2a. Create ECR Repository

In [ ]:
ecr = boto3.client("ecr", region_name=region)

try:
    ecr.create_repository(repositoryName=IMAGE_NAME)
    print(f"✅ Created ECR repository: {IMAGE_NAME}")
except ecr.exceptions.RepositoryAlreadyExistsException:
    print(f"ℹ️  ECR repository '{IMAGE_NAME}' already exists")

### 2b. Build and Push

Run these commands in your **terminal**:

```bash
# 1. Build the Docker image
mlflow models build-docker -m runs:/<RUN_ID>/model -n wine-quality-model

# 2. Login to ECR
aws ecr get-login-password --region <REGION> | docker login --username AWS --password-stdin <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com

# 3. Tag the image for ECR
docker tag wine-quality-model:latest <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/wine-quality-model:latest

# 4. Push to ECR
docker push <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/wine-quality-model:latest
```

In [ ]:
# Print the exact commands with your values filled in
print("📋 Run these commands in your terminal:\n")
print(f"# 1. Build Docker image")
print(f"mlflow models build-docker -m runs:/{run_id}/model -n {IMAGE_NAME}")
print(f"\n# 2. Login to ECR")
print(f"aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com")
print(f"\n# 3. Tag for ECR")
print(f"docker tag {IMAGE_NAME}:latest {ECR_REPO}:latest")
print(f"\n# 4. Push to ECR")
print(f"docker push {ECR_REPO}:latest")

## Step 3: Deploy to SageMaker

Once the image is in ECR, you can deploy it as a SageMaker endpoint.

In [ ]:
# NOTE: This step requires the sagemaker package and IAM permissions.
# Uncomment sagemaker in requirements.txt first: pip install sagemaker

print("📝 SageMaker Deployment Steps:")
print("")
print("1. Create a SageMaker model from the ECR image")
print("2. Create an endpoint configuration")
print("3. Create the endpoint")
print("4. Wait for the endpoint to be InService")
print("5. Send predictions to the endpoint")
print("6. ⚠️  DELETE the endpoint when done!")
print("")
print("The MLFlow SageMaker integration handles all of this:")
print(f"""  
import mlflow.sagemaker

mlflow.sagemaker.deploy(
    model_uri=f"runs:/{run_id}/model",
    app_name="wine-quality-endpoint",
    region_name="{region}",
    mode="create",
    image_url="{ECR_REPO}:latest",
)
""")

## Step 4: Alternative — Deploy to EC2

For more control, you can run the Docker container on an EC2 instance:

```bash
# On your EC2 instance:
# 1. Install Docker
sudo yum install docker -y
sudo service docker start

# 2. Login to ECR
aws ecr get-login-password --region <REGION> | docker login --username AWS --password-stdin <ECR_URL>

# 3. Pull and run
docker pull <ECR_URL>/wine-quality-model:latest
docker run -p 80:8080 <ECR_URL>/wine-quality-model:latest
```

Your model will be available at `http://<EC2_PUBLIC_IP>/invocations`.

## 🧹 Step 5: CLEANUP — Delete AWS Resources

**⚠️ IMPORTANT: Always run this to avoid charges!**

In [ ]:
# Cleanup ECR repository
# Uncomment and run when done with this module

# ecr = boto3.client("ecr", region_name=region)
# try:
#     ecr.delete_repository(repositoryName=IMAGE_NAME, force=True)
#     print(f"✅ Deleted ECR repository: {IMAGE_NAME}")
# except Exception as e:
#     print(f"Error: {e}")

print("⚠️  Uncomment the cleanup code above and run when done!")
print("   Also delete any SageMaker endpoints if created.")

## 🔑 Key Takeaways

1. **MLFlow → Docker → ECR → SageMaker/EC2** is the standard AWS deployment pipeline
2. **`mlflow models build-docker`** creates a production-ready container
3. **ECR** stores your Docker images in AWS
4. **SageMaker** gives you managed endpoints with auto-scaling
5. **EC2** gives you more control but requires managing infrastructure
6. **ALWAYS cleanup** resources to avoid unexpected charges

---

## 🎓 Module 7 Complete!

**Next up**: Module 8 — Advanced Topics →